# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [15]:
class matrix:
    def __init__(self, n, m=None):
        if isinstance(n, list):
            matrix._check(n)
            self.data = [list(r) for r in n]
        else:
            self.data = [[0.0] * m for _ in range(n)]

    def _check(L):
        if len(L) == 0 or any(len(r) != len(L[0]) for r in L):
            raise ValueError("Every row must have the same number of columns.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            if isinstance(i, int) and isinstance(j, int):
                return self.data[i][j]
            if isinstance(i, int): i = slice(i, i + 1)
            if isinstance(j, int): j = slice(j, j + 1)
            return matrix([r[j] for r in self.data[i]])
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            self.data[key[0]][key[1]] = value
        else:
            self.assign(value)

    def assign(self, other):
        values = other.data if isinstance(other, matrix) else other
        matrix._check(values)
        if (len(values), len(values[0])) != self.shape():
            raise ValueError("Size mismatch.")
        self.data = [list(r) for r in values]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([self.data[n]])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [list(r) for r in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([r[m_0:m_1] for r in self.data[n_0:n_1]])

    def _check_shape(self, N):
        if self.shape() != N.shape():
            raise ValueError("Shapes don't match.")

    def scalarmul(self, c):
        return matrix([[c * x for x in r] for r in self.data])

    def add(self, N):
        self._check_shape(N)
        return matrix([[a + b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def sub(self, N):
        self._check_shape(N)
        return matrix([[a - b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def mat_mult(self, N):
        if self.shape()[1] != N.shape()[0]:
            raise ValueError("Inner dimensions don't match.")
        return matrix([[sum(a * b for a, b in zip(r, col)) for col in zip(*N.data)]
                       for r in self.data])

    def element_mult(self, N):
        self._check_shape(N)
        return matrix([[a * b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def equals(self, N):
        return self.data == N.data

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)
        return self.scalarmul(other)

    def __rmul__(self, c):
        return self.scalarmul(c)

    def __add__(self, N):
        return self.add(N)

    def __sub__(self, N):
        return self.sub(N)

    def __eq__(self, N):
        return self.equals(N)

    def __repr__(self):
        return f"matrix({self.data})"

  # Used Claude to help me build construct the getitem and setitem functions. Also used it to help fix a few errors that were mostly caused by typos or my own error.


In [8]:
Z = matrix(2, 3)
print(Z)
M = matrix([[1, 2, 3], [4, 5, 6]])
print(M)
try:
    matrix([[1, 2], [3]])
except ValueError as e:
    print("Bad matrix caught:", e)

print(M[1][2], M[1, 2])
M[0, 1] = 20
M[0][2] = 30
print(M)

M_1 = matrix(2, 2)
M_2 = matrix([[1, 2], [3, 4]])
M_1[:] = M_2
print(M_1)
M_1[:] = [[5, 6], [7, 8]]
print(M_1)
try:
    M_1[:] = matrix(3, 3)
except ValueError as e:
    print("Size mismatch caught:", e)

# Used Claude here to help me test the first part of the function as I had some trouble trying to test it

matrix([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
matrix([[1, 2, 3], [4, 5, 6]])
Bad matrix caught: Every row must have the same number of columns.
6 6
matrix([[1, 20, 30], [4, 5, 6]])
matrix([[1, 2], [3, 4]])
matrix([[5, 6], [7, 8]])
Size mismatch caught: Size mismatch.


In [9]:


A = matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
print("shape:", A.shape())
print("transpose:", A.transpose())
print("row(1):", A.row(1), A.row(1).shape())
print("column(2):", A.column(2), A.column(2).shape())
print("to_list:", A.to_list())
print("block(1,3,0,2):", A.block(1, 3, 0, 2))
print("A[1:3]:", A[1:3])
print("A[1:3, 0:2]:", A[1:3, 0:2])
print("A[:, 1]:", A[:, 1])

shape: (4, 3)
transpose: matrix([[1, 4, 7, 10], [2, 5, 8, 11], [3, 6, 9, 12]])
row(1): matrix([[4, 5, 6]]) (1, 3)
column(2): matrix([[3], [6], [9], [12]]) (4, 1)
to_list: [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]
block(1,3,0,2): matrix([[4, 5], [7, 8]])
A[1:3]: matrix([[4, 5, 6], [7, 8, 9]])
A[1:3, 0:2]: matrix([[4, 5], [7, 8]])
A[:, 1]: matrix([[2], [5], [8], [11]])


In [17]:


def constant(n, m, c):
    return matrix([[float(c)] * m for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    I = zeros(n, n)
    for i in range(n):
        I[i, i] = 1.0
    return I

In [11]:


print(constant(2, 3, 7))
print(zeros(2, 2))
print(ones(2, 2))
print(eye(3))

matrix([[7.0, 7.0, 7.0], [7.0, 7.0, 7.0]])
matrix([[0.0, 0.0], [0.0, 0.0]])
matrix([[1.0, 1.0], [1.0, 1.0]])
matrix([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])


In [12]:


M = matrix([[1, 2], [3, 4]])
N = matrix([[5, 6], [7, 8]])
R = matrix([[1, 0, 2], [0, 1, 3]])

print("scalarmul:", M.scalarmul(3))
print("add:", M.add(N))
print("sub:", M.sub(N))
print("mat_mult:", M.mat_mult(N))
print("element_mult:", M.element_mult(N))
print("equals:", M.equals(N), M.equals(matrix([[1, 2], [3, 4]])))

try:
    M.add(R)
except ValueError as e:
    print("add caught:", e)
try:
    R.mat_mult(M)
except ValueError as e:
    print("mat_mult caught:", e)

scalarmul: matrix([[3, 6], [9, 12]])
add: matrix([[6, 8], [10, 12]])
sub: matrix([[-4, -4], [-4, -4]])
mat_mult: matrix([[19, 22], [43, 50]])
element_mult: matrix([[5, 12], [21, 32]])
equals: False True
add caught: Shapes don't match.
mat_mult caught: Inner dimensions don't match.


In [13]:
# Number 5

print("2*M =", 2 * M)
print("M*2 =", M * 2)
print("M+N =", M + N)
print("M-N =", M - N)
print("M*N =", M * N)
print("M==N:", M == N)

2*M = matrix([[2, 4], [6, 8]])
M*2 = matrix([[2, 4], [6, 8]])
M+N = matrix([[6, 8], [10, 12]])
M-N = matrix([[-4, -4], [-4, -4]])
M*N = matrix([[19, 22], [43, 50]])
M==N: False


In [14]:

# Number 6
A = matrix([[1, 2], [3, 4]])
B = matrix([[0, 1], [5, -2]])
C = matrix([[2, 0], [1, 3]])
I = eye(2)

print("(AB)C =", (A * B) * C, " A(BC) =", A * (B * C), " equal:", (A * B) * C == A * (B * C))
print("A(B+C) =", A * (B + C), " AB+AC =", A * B + A * C, " equal:", A * (B + C) == A * B + A * C)
print("AB =", A * B, " BA =", B * A, " equal:", A * B == B * A)
print("AI =", A * I, " equal to A:", A * I == A)

(AB)C = matrix([[17, -9], [35, -15]])  A(BC) = matrix([[17, -9], [35, -15]])  equal: True
A(B+C) = matrix([[14, 3], [30, 7]])  AB+AC = matrix([[14, 3], [30, 7]])  equal: True
AB = matrix([[10, -3], [20, -5]])  BA = matrix([[3, 4], [-1, 2]])  equal: False
AI = matrix([[1.0, 2.0], [3.0, 4.0]])  equal to A: True
